In [ ]:
# PIPELINE:
#
# [dados brutos]
#      ↓ normalização (estatística + álgebra linear)
# [dados padronizados]
#      ↓ PCA (covariância → autovetores → SVD)
# [representação reduzida]
#      ↓ embeddings (produto interno, similaridade)
# [espaço semântico]
#      ↓ rede neural (transformações lineares + não-linearidade)
#      ↓ gradiente descendente + backpropagation
# [modelo treinado]
#      ↓ atenção (produto interno escalado)
# [previsão com contexto]
#
# Cada seta é um commit deste repositório.
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

np.random.seed(42)

# ============================================================
# ETAPA 1 — Dados brutos: dataset sintético com estrutura real
# ============================================================
# Simula um dataset de classificação com:
# - 3 classes bem definidas mas com overlap
# - features em escalas diferentes
# - correlação entre algumas features

n_por_classe = 150
n_features = 6

# centros das 3 classes no espaço latente
centros = np.array([
    [2., 2., 0., 0., 1., 0.],
    [-2., 1., 1., 0., 0., 1.],
    [0., -2., 0., 1., -1., -1.],
])

# matriz de covariância compartilhada com correlação
cov_base = np.eye(n_features) * 0.8
cov_base[0, 1] = cov_base[1, 0] = 0.5
cov_base[2, 3] = cov_base[3, 2] = 0.4

X_partes = []
y_partes = []
for i, centro in enumerate(centros):
    X_c = np.random.multivariate_normal(centro, cov_base, n_por_classe)
    # features em escalas diferentes
    X_c[:, 0] *= 10.
    X_c[:, 3] *= 0.1
    X_partes.append(X_c)
    y_partes.append(np.full(n_por_classe, i))

X_raw = np.vstack(X_partes)
y = np.concatenate(y_partes).astype(int)
n_total = len(y)

print("=== ETAPA 1: DADOS BRUTOS ===")
print(f"Shape: {X_raw.shape}")
print(f"Médias por feature: {X_raw.mean(axis=0).round(2)}")
print(f"Desvios padrão:     {X_raw.std(axis=0).round(2)}")
print(f"→ escalas muito diferentes: feature 0 ≈ 10×, feature 3 ≈ 0.1×")

# ============================================================
# ETAPA 2 — Normalização: padronização Z-score
# ============================================================
# Commit 01: normas e escalas

mu = X_raw.mean(axis=0)
sigma = X_raw.std(axis=0)
X_std = (X_raw - mu) / sigma

print("\n=== ETAPA 2: NORMALIZAÇÃO (Z-SCORE) ===")
print(f"Médias após norm:   {X_std.mean(axis=0).round(6)}")
print(f"Desvios após norm:  {X_std.std(axis=0).round(6)}")
print(f"→ todas as features com média 0 e desvio 1")

# ============================================================
# ETAPA 3 — PCA: redução de 6 → 3 dimensões
# ============================================================
# Commits 03 e 05: covariância, autovetores, projeção

C = X_std.T @ X_std / (n_total - 1)
vals_pca, vecs_pca = np.linalg.eigh(C)
idx_pca = np.argsort(vals_pca)[::-1]
vals_pca = vals_pca[idx_pca]
vecs_pca = vecs_pca[:, idx_pca]

variancia_explicada = vals_pca / vals_pca.sum()
variancia_acum = np.cumsum(variancia_explicada)
k_pca = np.argmax(variancia_acum >= 0.85) + 1   # capturar 85% da variância

X_pca = X_std @ vecs_pca[:, :k_pca]

print(f"\n=== ETAPA 3: PCA ===")
print(f"Variância por componente: {(variancia_explicada*100).round(1)}")
print(f"Variância acumulada:      {(variancia_acum*100).round(1)}")
print(f"k escolhido (≥85%): {k_pca} componentes")
print(f"Shape após PCA: {X_pca.shape}")

# ============================================================
# ETAPA 4 — Matriz de similaridade: estrutura dos dados
# ============================================================
# Commit 01: produto interno e cosseno

def cos_sim_matrix(X):
    norms = np.linalg.norm(X, axis=1, keepdims=True) + 1e-8
    X_norm = X / norms
    return X_norm @ X_norm.T

S = cos_sim_matrix(X_pca)
# similaridade média intra-classe vs inter-classe
sim_intra = np.mean([S[i,j] for i in range(n_total) for j in range(n_total)
                     if y[i] == y[j] and i != j])
sim_inter = np.mean([S[i,j] for i in range(n_total) for j in range(n_total)
                     if y[i] != y[j]])
print(f"\n=== ETAPA 4: ESTRUTURA VETORIAL ===")
print(f"Similaridade intra-classe: {sim_intra:.3f}")
print(f"Similaridade inter-classe: {sim_inter:.3f}")
print(f"Razão: {sim_intra/abs(sim_inter):.2f}× — dados têm estrutura!")

# ============================================================
# ETAPA 5 — Rede neural: classificação
# ============================================================
# Commits 07, 08, 10: GD, backprop, geometria de redes

def softmax_rows(X):
    X_s = X - X.max(axis=1, keepdims=True)
    ex = np.exp(X_s)
    return ex / ex.sum(axis=1, keepdims=True)

def cross_entropy(y_pred, y_true, n_classes):
    n = len(y_true)
    log_pred = np.log(y_pred[np.arange(n), y_true] + 1e-8)
    return -log_pred.mean()

class MLP_Classificador:
    def __init__(self, dims, seed=42):
        np.random.seed(seed)
        self.dims = dims
        self.L = len(dims) - 1
        self.W = [np.random.randn(dims[i+1], dims[i]) * np.sqrt(2./dims[i])
                  for i in range(self.L)]
        self.b = [np.zeros(dims[i+1]) for i in range(self.L)]

    def forward(self, X):
        a = X
        cache = [X]
        zs = []
        for i, (W, b) in enumerate(zip(self.W, self.b)):
            z = a @ W.T + b
            zs.append(z)
            a = np.maximum(0, z) if i < self.L-1 else softmax_rows(z)
            cache.append(a)
        return a, cache, zs

    def treinar(self, X, y, alpha=0.01, n_iter=500, batch=64):
        n = len(y)
        n_classes = self.dims[-1]
        hist_loss, hist_acc = [], []

        for it in range(n_iter):
            idx = np.random.choice(n, batch, replace=False)
            Xb, yb = X[idx], y[idx]

            y_pred, cache, zs = self.forward(Xb)
            loss = cross_entropy(y_pred, yb, n_classes)

            # backward
            delta = y_pred.copy()
            delta[np.arange(batch), yb] -= 1.
            delta /= batch

            for l in reversed(range(self.L)):
                dW = delta.T @ cache[l]
                db = delta.sum(axis=0)
                self.W[l] -= alpha * dW
                self.b[l] -= alpha * db
                if l > 0:
                    delta = delta @ self.W[l]
                    delta *= (zs[l-1] > 0).astype(float)

            if it % 50 == 0:
                y_all, _, _ = self.forward(X)
                acc = np.mean(y_all.argmax(axis=1) == y)
                hist_loss.append(cross_entropy(y_all, y, n_classes))
                hist_acc.append(acc)

        return hist_loss, hist_acc

    def predict(self, X):
        out, _, _ = self.forward(X)
        return out.argmax(axis=1)

# treinar
modelo = MLP_Classificador([k_pca, 32, 16, 3])
hist_loss, hist_acc = modelo.treinar(X_pca, y, alpha=0.02, n_iter=500)
acc_final = np.mean(modelo.predict(X_pca) == y)
print(f"\n=== ETAPA 5: REDE NEURAL ===")
print(f"Acurácia final: {acc_final:.3f}")

# ============================================================
# ETAPA 6 — Atenção: contextualizar as previsões
# ============================================================
# Commit 11: produto interno escalado

# usar as últimas ativações como "tokens"
_, cache_final, _ = modelo.forward(X_pca[:20])
repr_final = cache_final[-2]   # última camada oculta, primeiros 20 exemplos

d_k_integ = repr_final.shape[1]
W_Q_integ = np.random.randn(d_k_integ, d_k_integ) * 0.1
W_K_integ = np.random.randn(d_k_integ, d_k_integ) * 0.1

Q_integ = repr_final @ W_Q_integ
K_integ = repr_final @ W_K_integ
scores_integ = Q_integ @ K_integ.T / np.sqrt(d_k_integ)
atencao_integ = softmax(scores_integ)

def softmax(x, axis=-1):
    x_s = x - x.max(axis=axis, keepdims=True)
    ex = np.exp(x_s)
    return ex / ex.sum(axis=axis, keepdims=True)

atencao_integ = softmax(scores_integ)
print(f"\n=== ETAPA 6: ATENÇÃO ===")
print(f"Matriz de atenção shape: {atencao_integ.shape}")
print(f"Pesos de atenção para exemplo 0: {atencao_integ[0, :5].round(3)} ...")

# ============================================================
# ETAPA 7 — VISUALIZAÇÃO FINAL: o pipeline completo
# ============================================================

fig = plt.figure(figsize=(18, 12))
fig.suptitle("Commit 12 — Pipeline Completo: Dado → Vetor → Modelo", fontsize=14)
gs = gridspec.GridSpec(3, 4, figure=fig, hspace=0.4, wspace=0.35)

cores_classes = ['#534AB7', '#D85A30', '#0F6E56']
labels_classes = ['classe 0', 'classe 1', 'classe 2']

# 1. Dados brutos (features 0 e 1)
ax1 = fig.add_subplot(gs[0, 0])
for i in range(3):
    mask = y == i
    ax1.scatter(X_raw[mask, 0], X_raw[mask, 1], s=8, alpha=0.4,
                color=cores_classes[i], label=labels_classes[i])
ax1.set_title("1. Dados brutos\n(escalas diferentes)", fontsize=9)
ax1.grid(True, alpha=0.2); ax1.legend(fontsize=6)

# 2. Dados normalizados
ax2 = fig.add_subplot(gs[0, 1])
for i in range(3):
    mask = y == i
    ax2.scatter(X_std[mask, 0], X_std[mask, 1], s=8, alpha=0.4,
                color=cores_classes[i])
ax2.set_title("2. Normalizado (Z-score)\nescalas uniformes", fontsize=9)
ax2.grid(True, alpha=0.2)

# 3. PCA 2D
ax3 = fig.add_subplot(gs[0, 2])
for i in range(3):
    mask = y == i
    ax3.scatter(X_pca[mask, 0], X_pca[mask, 1], s=8, alpha=0.4,
                color=cores_classes[i])
ax3.set_title(f"3. PCA ({k_pca} PCs)\n{variancia_acum[k_pca-1]*100:.0f}% variância", fontsize=9)
ax3.grid(True, alpha=0.2)

# 4. Variância explicada
ax4 = fig.add_subplot(gs[0, 3])
ax4.bar(range(1, n_features+1), variancia_explicada*100,
        color='#534AB7', alpha=0.8)
ax4.plot(range(1, n_features+1), variancia_acum*100, 'ro-', lw=1.5, ms=4)
ax4.axhline(85, color='red', linestyle='--', lw=0.8, alpha=0.5)
ax4.set_xlabel("componente"); ax4.set_ylabel("var. explicada (%)")
ax4.set_title("3b. Variância por PC\n(linha = acumulada)", fontsize=9)
ax4.grid(True, alpha=0.2, axis='y')

# 5. Matriz de similaridade (amostra de 30)
ax5 = fig.add_subplot(gs[1, 0])
idx_sample = np.concatenate([np.where(y==i)[0][:10] for i in range(3)])
im5 = ax5.imshow(S[np.ix_(idx_sample, idx_sample)], cmap='RdYlGn',
                  vmin=-1, vmax=1, aspect='auto')
ax5.set_title("4. Matriz de similaridade\n(30 amostras, 10/classe)", fontsize=9)
plt.colorbar(im5, ax=ax5, shrink=0.7)

# 6. Loss e acurácia
ax6 = fig.add_subplot(gs[1, 1])
iters = range(0, 500, 50)
ax6.plot(list(iters)[:len(hist_loss)], hist_loss, color='#D85A30', lw=2, label='loss')
ax6b = ax6.twinx()
ax6b.plot(list(iters)[:len(hist_acc)], [a*100 for a in hist_acc],
          color='#534AB7', lw=2, label='acurácia %')
ax6.set_xlabel("iteração"); ax6.set_ylabel("loss", color='#D85A30')
ax6b.set_ylabel("acurácia (%)", color='#534AB7')
ax6.set_title("5. Treinamento da rede", fontsize=9)
ax6.grid(True, alpha=0.2)

# 7. Representação final da rede (2D via PCA)
_, cache_all, _ = modelo.forward(X_pca)
repr_hidden = cache_all[-2]
repr_h_c = repr_hidden - repr_hidden.mean(axis=0)
_, _, Vt_h = np.linalg.svd(repr_h_c, full_matrices=False)
repr_h_2d = repr_h_c @ Vt_h[:2].T

ax7 = fig.add_subplot(gs[1, 2])
for i in range(3):
    mask = y == i
    ax7.scatter(repr_h_2d[mask, 0], repr_h_2d[mask, 1], s=8, alpha=0.4,
                color=cores_classes[i], label=labels_classes[i])
ax7.set_title("5b. Representação oculta\ndados separáveis linearmente", fontsize=9)
ax7.grid(True, alpha=0.2); ax7.legend(fontsize=6)

# 8. Atenção final
ax8 = fig.add_subplot(gs[1, 3])
im8 = ax8.imshow(atencao_integ, cmap='Blues', vmin=0, vmax=1, aspect='auto')
ax8.set_title("6. Pesos de atenção\n(20 amostras da repr. oculta)", fontsize=9)
ax8.set_xlabel("key"); ax8.set_ylabel("query")
plt.colorbar(im8, ax=ax8, shrink=0.7)

# 9. Resumo do pipeline (texto)
ax9 = fig.add_subplot(gs[2, :])
ax9.axis('off')
pipeline_texto = """
PIPELINE COMPLETO  —  do dado bruto ao modelo com atenção

  [dado bruto R⁶]
      → commit 01/02:  normalização Z-score, escalas vetoriais, transformação linear
      → commit 03/05:  PCA via covariância + autovetores  →  [repr. R³, 85% variância]
      → commit 01:     matriz de similaridade cosseno  →  [estrutura semântica verificada]
      → commit 07/08:  gradiente descendente + backpropagation  →  [rede MLP treinada, acc={:.1f}%]
      → commit 10:     representação oculta linearmente separável
      → commit 11:     atenção  QKᵀ/√d  →  [pesos contextuais sobre a representação]

  Cada seta é um commit. Cada commit é álgebra linear aplicada.
""".format(acc_final*100)

ax9.text(0.01, 0.5, pipeline_texto, transform=ax9.transAxes,
         fontsize=10, verticalalignment='center', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='#f5f5f0', alpha=0.5))

plt.savefig('assets/12_pipeline_integrador.png', dpi=150, bbox_inches='tight')
plt.show()

=== ETAPA 1: DADOS BRUTOS ===
Shape: (450, 6)
Médias por feature: [-1.03  0.32  0.33  0.03  0.03 -0.02]
Desvios padrão:     [18.41  1.81  1.04  0.1   1.22  1.22]
→ escalas muito diferentes: feature 0 ≈ 10×, feature 3 ≈ 0.1×

=== ETAPA 2: NORMALIZAÇÃO (Z-SCORE) ===
Médias após norm:   [ 0. -0.  0. -0.  0.  0.]
Desvios após norm:  [1. 1. 1. 1. 1. 1.]
→ todas as features com média 0 e desvio 1

=== ETAPA 3: PCA ===
Variância por componente: [36.2 26.7 18.5  8.6  6.5  3.5]
Variância acumulada:      [ 36.2  62.9  81.4  90.   96.5 100. ]
k escolhido (≥85%): 4 componentes
Shape após PCA: (450, 4)

=== ETAPA 4: ESTRUTURA VETORIAL ===
Similaridade intra-classe: 0.578
Similaridade inter-classe: -0.289
Razão: 2.00× — dados têm estrutura!

=== ETAPA 5: REDE NEURAL ===
Acurácia final: 0.984


NameError: name 'softmax' is not defined